## Data preparation

We intend to use spectral embedding on this dataset to convert it into a set of continuous latent variables.  
Often this algorithm requires a preliminary unsupervised learning to identify related data points (typically using Random Forests) but that is not necessary here as we already have our data in the form of user-to-song associations which can be viewed as edges in a graph.

However we still need to convert the source table to a sparse matrix, which we will implement in this notebook.

Import the essential libraries that we need

In [1]:
from zipfile import ZipFile, ZipInfo
from datetime import datetime
import os
import csv

Set the working directory to the project root to simplify access to the data and other Python modules

In [2]:
current_dir = os.getcwd()
project_directory, notebooks_subdirectory = os.path.split(current_dir)
if notebooks_subdirectory == "notebooks":
    os.chdir(project_directory)
    print(f"Changed working directory to project root: {project_directory}")
else:
    project_directory = current_dir
# Confirm the new current directory
os.getcwd()

Changed working directory to project root: c:\Users\dave4\vscode-projects\spotify-playlists


'c:\\Users\\dave4\\vscode-projects\\spotify-playlists'

Also add our local modules to the import path

In [3]:
data_directory = os.path.join(project_directory, "data")
if project_directory not in os.sys.path:
    os.sys.path.append(project_directory)

Define some helper functions for reading the source data.  
This dataset is fairly large, so we stream it one row at a time. We will use this strategy frequently in this project.

In [4]:
archive_path = os.path.normpath(os.path.join(data_directory, 'archive.zip'))

def is_odd(n):
    return n % 2 == 1
    
def decoded_line_generator(file):
    for line in file:
        line_str = line.decode('utf-8')
        if line_str.count('"') != 8:
            # Song titles with double quotes cause CSV parsing issues as they have not been escaped.
            # Re-encode them to make the CSV parser happy.
            # This will not work in the general case, but is sufficient for this dataset.
            line_str = line_str[1:-1]  # Remove leading and trailing quotes
            line_str = line_str.replace('@', '@0')
            line_str = line_str.replace('","', '@1') # Unambiguously mark the field separators
            line_str = line_str.replace('"', '@2')   # Remaining quotes are part of the title
            # Reassemble the line but double up any quotes in the title
            line_str = line_str.replace('@2', '""')
            line_str = line_str.replace('@1', '","')
            line_str = line_str.replace('@0', '@')
            line_str = f'"{line_str}"'  # Restore leading and trailing quotes
        yield line_str

def process_csv(row_consumer):
    with ZipFile(archive_path, 'r') as archive:
        with archive.open(name='spotify_dataset.csv', mode='r') as file:
            csv_reader = csv.reader(decoded_line_generator(file))
            next(csv_reader)  # Skip header
            for row in csv_reader:
                if not row_consumer.handle_row(row):
                    break


Check that everything is working

In [5]:
class Consumer:
    row_count: int
   
    def __init__(self):
        self.row_count = 0

    def handle_row(self, row: any) -> bool:
        if self.row_count < 20: # Print the first 20 rows to visually check the data
            print(row)
        self.row_count += 1
        return True

consumer = Consumer()
process_csv(consumer) # Read the entire CSV file. Can take a while (up to 40 seconds)
consumer.row_count    # Confirm total number of rows processed (should be 12,901,979)

['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello', '(The Angels Wanna Wear My) Red Shoes', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello & The Attractions', "(What's So Funny 'Bout) Peace, Love And Understanding", 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Tiffany Page', '7 Years Too Late', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello & The Attractions', 'Accidents Will Happen', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Elvis Costello', 'Alison', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Lissie', 'All Be Okay', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Paul McCartney', 'Band On The Run', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Joe Echo', 'Beautiful', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Paul McCartney', 'Blackbird - Live at CitiField, NYC - Digital Audio', 'HARD ROCK 2010']
['9cc0cfd4d7d7885102480dd99e7a90d6', 'Lissie', 'Bright Side', 'HARD RO

12902577

In [6]:
# Discard temporary data
del consumer

The next stage is to apply a label encoding scheme to replace our string attributes with integers (which will later be used as indices into a matrix).

In [7]:
from LabelEncoding import LabelEncoding

class Consumer:
    def __init__(self, listens_path, songs_path):
        self.user_encoder = LabelEncoding()
        self.song_encoder = LabelEncoding()
        self.listens_path = listens_path
        self.songs_path = songs_path
        self.listens_file = None
        self.unique_listens = set()

    def __enter__(self):
        self.listens_file = open(self.listens_path, 'w', encoding='utf-8')
        return self

    def __exit__(self, _exc_type, _exc_value, _traceback):
        self.song_encoder.export_csv(self.songs_path, fields=['artistname', 'trackname'])
        if self.listens_file:
            self.listens_file.close()

    def handle_row(self, row: any) -> bool:
        if len(row) != 4:
            raise ValueError(f"Unexpected number of columns in row #{len(self.user_encoder.mapping)}: {row}")
        user_id, artist_name, track_name, playlist_name = row
        user_index = self.user_encoder.get_or_create_id(user_id)
        song_index = self.song_encoder.get_or_create_id((artist_name, track_name))
        if (user_index, song_index) in self.unique_listens:
            return True  # Skip duplicate listens
        self.unique_listens.add((user_index, song_index))
        self.listens_file.write(f"{user_index},{song_index}\n")
        return True

listens_path = os.path.join(data_directory, 'listens.csv')
songs_path = os.path.join(data_directory, 'songs.csv')



In [8]:
with Consumer(listens_path, songs_path) as consumer:
    process_csv(consumer) # Read the entire CSV file. Can take a while (up to 2 minutes)
    print(f"Total unique users: {len(consumer.user_encoder.mapping)}")
    print(f"Total unique songs: {len(consumer.song_encoder.mapping)}")
    print(f"Total repeated songs: {len(consumer.song_encoder.repeated_values)}") # Songs with at least two unique listeners

Total unique users: 15918
Total unique songs: 2824004
Total repeated songs: 1221147


In [9]:
from MostPopular import extract_top_songs

popular_songs_path = os.path.join(data_directory, 'popular_songs.csv')
popular_listens_path = os.path.join(data_directory, 'popular_listens.csv')

extract_top_songs(listens_path, songs_path, popular_listens_path, popular_songs_path, min_listens=200)
